## Programming for Data Science 
### Project Notebook: "Where should I live?" 
#### Group Members:
- Afonso Fernandes / 20241710
- Lourenço Lima / 20241711
- Pedro Jorge / 20241819
- David Morais / 20241759
## Project Repository
GitHub Repository:  
https://github.com/afonsolince06/-Where-should-I-live-PDS-Project

### Introduction

In this part of the project, the goal is to build an interactive map of Europe that allows users to explore key information about major European cities. The task combines web scraping, data integration, and geospatial visualization to create an informative and interactive tool.

To accomplish this, we will:

-Scrape the geographical coordinates of each city directly from the Wikipedia Main Page, ensuring accuracy and consistency with the provided dataset.

-Match the scraped coordinates with the dataset entries so that each city is correctly assigned to its corresponding country, population, average salary, and cost of living.

-Use the cleaned and enriched dataset to construct an interactive map of Europe, where each city can be clicked or hovered over to display its relevant information.

By the end of this section, we will have a fully functional map that visually represents European cities and provides meaningful insights through an intuitive interface. This builds on the skills developed earlier in the project and introduces new concepts in geospatial data handling and visualization.

#### Import essential libraries and define an alias for them

In [1]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from datetime import datetime
from bs4 import BeautifulSoup       # process html
from selenium import webdriver      # automate web browser interaction
from selenium.webdriver.common.by import By #specify specify how to locate elements on a web page.
import requests                    # make requests to fetch web pages
import time
import ipywidgets as widgets
from IPython.display import display
import re

SyntaxError: invalid syntax (2688915260.py, line 4)

In [ ]:
full_data=pd.read_csv('city_data_with_coords.csv') 
#Load the data with both information from the first and second notebook, such as average monthly salary and the coordinates of each city
full_data.head()

In [ ]:
raw_langs = full_data['Main Spoken Languages'].astype(str).str.split(',')
flat_list = [lang.strip() for sublist in raw_langs if isinstance(sublist, list) for lang in sublist]# Flatten the list (turn list of lists into one big list) and strip whitespace
languages = sorted(list(set(flat_list))) #Get unique values for the list and sort them
print(f"Loaded {len(languages)} unique individual languages.")
print(languages)

Web Scrapping Safety, Health and  Index

In [ ]:
def get_extra_index():

In [ ]:
def dms_to_decimal(dms_coords):
    # Check if the value is already a float/int (already converted)
    if isinstance(dms_coords, (float, int)):
        return dms_coords
        
    try:
        dms_parts = re.split('[°′″NEW]', str(dms_coords))
        degrees = float(dms_parts[0]) 
        minutes = float(dms_parts[1]) if len(dms_parts) > 1 and dms_parts[1] else 0 
        seconds = float(dms_parts[2]) if len(dms_parts) > 2 and dms_parts[2] else 0 

        decimal_coords = degrees + (minutes / 60) + (seconds / 3600) 

        if 'S' in dms_coords or 'W' in dms_coords:
            decimal_coords = -decimal_coords
            
        return decimal_coords
    except Exception as e:
        return None # Handle errors gracefully

# Apply conversion immediately to the master dataframe
# (Make sure full_data is loaded before running this)
full_data['Latitude'] = full_data['Latitude'].apply(dms_to_decimal)
full_data['Longitude'] = full_data['Longitude'].apply(dms_to_decimal)

In [ ]:
# --- 1. SETUP WIDGETS (Inputs) ---
min_salary_slider = widgets.IntSlider(
    value=2000, min=500, max=10000, step=100, 
    description='Min Salary (€):', style={'description_width': 'initial'}, layout=widgets.Layout(width='60%')
)

max_cost_slider = widgets.IntSlider(
    value=1500, min=500, max=5000, step=100, 
    description='Max Cost of Living (€):', style={'description_width': 'initial'}, layout=widgets.Layout(width='60%')
)

max_unemployer_slider = widgets.FloatSlider(
    value=5, min=0, max=25, step=0.5, 
    description='Max Unemployment Rate (%):', style={'description_width': 'initial'}, layout=widgets.Layout(width='60%')
)

# --- NEW: CHECKBOX LIST CREATION ---
# 1. Create a list of Checkbox widgets (one for each language)
checkbox_objects = []
for lang in languages:
    checkbox_objects.append(widgets.Checkbox(value=False, description=lang, indent=False))

# 2. Put them inside a container (VBox)
# 3. Add Layout to make it scrollable so it doesn't take up the whole screen
checkbox_container = widgets.VBox(
    children=checkbox_objects, 
    layout=widgets.Layout(
        width='100%', 
        height='180px',       # Fixed height
        overflow_y='scroll',  # Adds a scrollbar
        border='1px solid black', 
        padding='5px'
    )
)

# Label for the box
lang_label = widgets.Label("Select Languages (Scroll to see more):")

# Button to trigger the update
run_button = widgets.Button(description="Find My City", button_style='success')
output_area = widgets.Output()

# --- 2. DEFINE THE INTERACTION ---
def on_button_click(b):
    with output_area:
        output_area.clear_output()
        
        # Get values from sliders
        user_min_salary = min_salary_slider.value
        user_max_cost = max_cost_slider.value
        user_max_unemployment= max_unemployer_slider.value
        # --- NEW: EXTRACT SELECTED CHECKBOXES ---
        # Loop through our list of widgets and check which ones are ticked
        selected_langs = [box.description for box in checkbox_objects if box.value]
        
        # Validation: If nothing is checked, warn the user (or treat as "All")
        if not selected_langs:
            print("⚠️ Please select at least one language from the list above.")
            return
        
        def check_lang_match(city_lang_str):
            if not isinstance(city_lang_str, str): return False
            # Split the city's languages into a list
            city_langs_list = [l.strip() for l in city_lang_str.split(',')]
            # Return True if there is an overlap (intersection)
            return any(lang in city_langs_list for lang in selected_langs)

        # Create a boolean mask using our new check function
        language_mask = full_data['Main Spoken Languages'].apply(check_lang_match)
        
        # Filter the Master DataFrame (Use the one created in Step 2)
        # Note: Make sure 'master_df' exists before running this
        results = full_data[
            (full_data['Average Monthly Salary'] >= user_min_salary) &
            (full_data['Average Cost of Living'] <= user_max_cost) &
            (full_data['Unemployment Rate'] <= user_max_unemployment) &
            (language_mask)
        ]
        
        if results.empty:
            print("No cities match your criteria! Try lowering your standards or asking for a raise 😉")
        else:
            print(f"Found {len(results)} cities matching your criteria:")
            display(results.tail(15)) # Show table
            
            # Create Map (Using your code from testar_web_scrapping.ipynb)
            fig_map = px.scatter_mapbox(
                results,
                lat="Latitude",
                lon="Longitude",
                hover_name="City",
                size="Population", # Optional
                color="Average Monthly Salary",
                color_continuous_scale="Turbo",
                zoom=3,
                mapbox_style="open-street-map",
                title="Recommended Cities"
            )
            fig_map.show()

# --- 3. DISPLAY EVERYTHING ---
run_button.on_click(on_button_click)
ui = widgets.VBox([
    min_salary_slider, 
    max_cost_slider,
    max_unemployer_slider, 
    lang_label, 
    checkbox_container, 
    run_button
])

display(ui, output_area)